# Project 01: Student Support and Course Feedback Assistant
## Training Notebook (Steps A to E)

**Is notebook me kya hai?** Data load karna, 3 charts, Decision Tree + Baseline, KMeans, CNN, aur NLP (VADER).
Aakhir me saare models `models/` folder me save ho jate hain, taake app unhe sirf load kare (dobara train na kare).

**Kaise chalana hai:** Colab me ye notebook kholein aur cells ko **upar se neeche, ek ek kar ke** chalayein (Shift + Enter).

**Aham usool:**
- Test data sirf **aakhir me ek dafa** istemal hota hai.
- `record_id`, `split` aur `needs_support` model ke **input nahi** hain.
- Reference labels (`sentiment_label`) kabhi nahi badalne.
- Sirf apne **asli numbers** report me likhein.

## Step 0: Setup (ZIP upload aur packages)
Agle cell me **`Project_01.zip`** upload karni hai. Is ZIP ke andar `Project_01/` folder hai jis me `data/`, `nltk_data/`, `sample_images/` waghera sab maujood hain.

In [ ]:
# Cell 1: ZIP upload (sirf Colab me chalta hai)
from google.colab import files
files.upload()   # "Choose files" par click karein aur Project_01.zip chunein

In [ ]:
# Cell 2: ZIP kholna aur extra packages install karna
from pathlib import Path
import zipfile

# /content me jo bhi Project_01 wali ZIP upload hui hai, sab se nayi wali uthao
zips = sorted(Path("/content").glob("Project_01*.zip"), key=lambda p: p.stat().st_mtime)
assert zips, "Project_01.zip nahi mili. Pehle Cell 1 chalakar ZIP upload karein."
with zipfile.ZipFile(zips[-1]) as archive:
    archive.extractall("/content")
print("Khol di gayi:", zips[-1].name)

%pip install -q nltk groq

## Step 1: Data load karna aur check karna
`PROJECT_DIR` project ka folder hai, `DATA` us ke andar `data/` folder. Ye dono naam aage sab jagah use honge.

In [ ]:
# Cell 3: Saari files load karna
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Project_01 folder dhoondna (Colab me ye /content/Project_01 hota hai)
ROOT = Path("/content")      # apne computer par chalayen to ROOT ko apne folder se badal dein
found = list(ROOT.glob("Project_01/data/main.csv")) or list(ROOT.glob("*/Project_01/data/main.csv"))
assert found, "Project_01/data/main.csv nahi mili. Cell 1 aur Cell 2 dobara chalayein."
PROJECT_DIR = found[0].parent.parent
DATA = PROJECT_DIR / "data"
print("Project folder:", PROJECT_DIR)

# models/ aur reports/ folder banana (pehle se hon to koi masla nahi)
(PROJECT_DIR / "models").mkdir(exist_ok=True)
(PROJECT_DIR / "reports").mkdir(exist_ok=True)

meta = json.loads((DATA / "meta.json").read_text())
main = pd.read_csv(DATA / "main.csv")
feedback = pd.read_csv(DATA / "feedback.csv")
images = np.load(DATA / "images.npz", allow_pickle=False)

print("Split ke hisaab se rows:")
print(main.groupby("split").size())
print()
print("Image arrays ki shape:", {name: images[name].shape for name in images.files})
print("Image classes:", meta["image_classes"])
main.head()

In [ ]:
# Cell 4: Data ki sehat check karna (khali values, ranges)
print("Khali (missing) values:", int(main.isna().sum().sum()))
print()

# meta.json me likhi ranges ke andar values hain ya nahi
for col, (low, high) in meta["numeric_ranges"].items():
    ok = main[col].between(low, high).all()
    print(f"{col}: range {low} se {high} -> {'theek' if ok else 'RANGE SE BAHAR'}")

for col, allowed in meta["category_values"].items():
    ok = main[col].isin(allowed).all()
    print(f"{col}: allowed {allowed} -> {'theek' if ok else 'GHALAT VALUE'}")

# Bunyadi tarteeb: train / validation / test alag alag
train = main[main["split"] == "train"]
val = main[main["split"] == "validation"]
test = main[main["split"] == "test"]
print()
print("train:", len(train), " validation:", len(val), " test:", len(test))

## Step 2: Teen charts (sirf training rows se)
1. Support label ki ginti
2. G2 (doosre period ke marks) ka histogram
3. Har `studytime` par G2 ka average

**Yaad rakhein:** chart me jo relation nazar aaye usay "wajah" mat likhein. Sirf "related hai" likhein.

In [ ]:
# Cell 5: Teen charts
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chart 1: target distribution
counts = train["needs_support"].value_counts().sort_index()
axes[0].bar(["0 (madad nahi)", "1 (madad chahiye)"], counts.values, color=["#4C78A8", "#E45756"])
axes[0].set_title("Support label ki ginti (train)")
axes[0].set_ylabel("Records")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha="center")

# Chart 2: ek feature ki distribution (G2)
axes[1].hist(train["G2"], bins=range(0, 22), color="#4C78A8", edgecolor="white")
axes[1].set_title("G2 (doosre period ke marks) ka histogram")
axes[1].set_xlabel("G2 (0 se 20)")
axes[1].set_ylabel("Records")

# Chart 3: feature aur target ka relation (studytime vs average G2)
avg_g2 = train.groupby("studytime")["G2"].mean()
axes[2].bar(avg_g2.index.astype(str), avg_g2.values, color="#59A14F")
axes[2].set_title("Har studytime par average G2")
axes[2].set_xlabel("studytime (1 = kam, 4 = zyada)")
axes[2].set_ylabel("Average G2")

plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/01_three_charts.png", dpi=150)
plt.show()

print(avg_g2.round(2).to_dict())

## Step 3: Supervised learning: Decision Tree + Baseline
- **Model:** Decision Tree (`max_depth=4`) jo `needs_support` predict karta hai.
- **Baseline:** `DummyClassifier`, jo hamesha sab se aam jawab deta hai. Aap ka model us se behtar hona chahiye.
- **Input features:** `G1, G2, studytime, failures, schoolsup, famsup, internet`.
- Model **train** par sikhta hai, **validation** par check hota hai, aur **test** par sirf aakhir me ek dafa.

In [ ]:
# Cell 6: Model banana aur train karna
import joblib
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

num = meta["numeric_features"]          # G1, G2, studytime, failures
cat = meta["categorical_features"]      # schoolsup, famsup, internet
features = num + cat
target = meta["target"]                 # needs_support

# OneHotEncoder: yes/no jaise alfaaz ko 0/1 columns me badalta hai
encode = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    remainder="passthrough")

supervised_model = make_pipeline(encode, DecisionTreeClassifier(max_depth=4, random_state=42))
supervised_model.fit(train[features], train[target])

# Baseline: hamesha "most frequent" class batata hai
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train[features], train[target])

# Validation par ek chhota sa check
from sklearn.metrics import accuracy_score, f1_score
val_pred = supervised_model.predict(val[features])
print("Validation accuracy:", round(accuracy_score(val[target], val_pred), 3))
print("Validation macro F1:", round(f1_score(val[target], val_pred, average="macro"), 3))

# Model save
joblib.dump(supervised_model, PROJECT_DIR / "models/supervised.joblib")
print("Saved: models/supervised.joblib")

In [ ]:
# Cell 7: FINAL test evaluation (sirf ek dafa chalayein)
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, ConfusionMatrixDisplay)

def tabular_metrics(model, df):
    pred = model.predict(df[features])
    y = df[target]
    return {
        "accuracy": round(accuracy_score(y, pred), 3),
        "macro_f1": round(f1_score(y, pred, average="macro"), 3),
        "class1_precision": round(precision_score(y, pred, zero_division=0), 3),
        "class1_recall": round(recall_score(y, pred, zero_division=0), 3),
        "confusion_matrix": confusion_matrix(y, pred).tolist(),
    }, pred

tree_metrics, test_pred = tabular_metrics(supervised_model, test)
base_metrics, _ = tabular_metrics(baseline, test)

print("Test me har class ke examples:", test[target].value_counts().sort_index().to_dict())
print()
print("DECISION TREE:", tree_metrics)
print("BASELINE     :", base_metrics)

# Confusion matrix ki tasveer
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(test[target], test_pred, ax=ax, colorbar=False)
ax.set_title("Decision Tree: test confusion matrix")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/02_tree_confusion.png", dpi=150)
plt.show()

# Paanch galat predictions (report me likhne ke liye)
errors = test.copy()
errors["predicted"] = test_pred
errors = errors[errors["predicted"] != errors[target]]
print()
print("Galat predictions ki tadaad:", len(errors))
errors_5 = errors[["record_id"] + features + [target, "predicted"]].head(5)
errors_5.to_csv(PROJECT_DIR / "reports/tree_errors.csv", index=False)
display(errors_5)

# Metrics file me save
(PROJECT_DIR / "reports/tabular_metrics.json").write_text(
    json.dumps({"tree": tree_metrics, "baseline": base_metrics}, indent=2))

In [ ]:
# Cell 8 (optional): Tree ne kaun se sawal seekhe? Samajhne ke liye
from sklearn.tree import export_text
names = list(supervised_model[:-1].get_feature_names_out())
print(export_text(supervised_model[-1], feature_names=names, max_depth=3))

## Step 4: KMeans (3 groups)
- Inputs: `G1, G2, studytime, failures`. Koi label istemal nahi hota.
- `StandardScaler` pehle sab features ko ek jaisi scale par laata hai.
- Group ka number (0, 1, 2) sirf ID hai. Group 0 "acha" ya "bura" nahi hota.
- Group ka naam tab dein jab **original units wale averages** dekh lein.

In [ ]:
# Cell 9: KMeans train karna aur groups dekhna
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_features = meta["cluster_features"]

cluster_model = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=42))
group_ids = cluster_model.fit_predict(train[cluster_features])

profiles = train[cluster_features].copy()
profiles["group"] = group_ids

summary = profiles.groupby("group").mean().round(2)
summary.insert(0, "students", profiles.groupby("group").size())
print("Har group ka size aur original units me average:")
display(summary)

In [ ]:
# Cell 10: Scatter plot (G1 vs G2, rang = group)
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(profiles["G1"], profiles["G2"], c=profiles["group"], cmap="viridis", alpha=0.7)
ax.set_xlabel("G1 (pehle period ke marks)")
ax.set_ylabel("G2 (doosre period ke marks)")
ax.set_title("KMeans: 3 learning profile groups (train)")
ax.legend(*scatter.legend_elements(), title="Group")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/03_kmeans_scatter.png", dpi=150)
plt.show()

In [ ]:
# Cell 11: Group descriptions ka MASAUDA banana, phir aap khud theek karein
# Group G2 ke average ke hisaab se tarteeb me rakhe gaye. Ye sirf draft hai, averages dekh kar edit karein!
order = summary["G2"].sort_values().index.tolist()   # sab se kam G2 se sab se zyada G2
draft_names = ["Lower previous marks", "Middle previous marks", "Higher previous marks"]

group_descriptions = {}
for rank, g in enumerate(order):
    row = summary.loc[g]
    group_descriptions[str(g)] = (
        f"{draft_names[rank]}: average G1 {row['G1']}, G2 {row['G2']}, "
        f"studytime {row['studytime']}, failures {row['failures']}.")
for k, v in group_descriptions.items():
    print("Group", k, "->", v)

(PROJECT_DIR / "models/group_descriptions.json").write_text(json.dumps(group_descriptions, indent=2))
joblib.dump(cluster_model, PROJECT_DIR / "models/clusters.joblib")
print("Saved: models/clusters.joblib aur models/group_descriptions.json")

# Naye record ke liye group sirf predict() se milta hai (dobara fit nahi!)
sample_record = pd.DataFrame([{"G1": 10, "G2": 11, "studytime": 2, "failures": 0}])
print("Misaal record ka group:", int(cluster_model.predict(sample_record[cluster_features])[0]))

## Step 5: Chhota CNN (haath se likhe digit)
- Image 28 x 28 x 1, aur 10 classes (0 se 9).
- Pixels pehle se 0 se 255 hain. Model khud `Rescaling(1./255)` karta hai, **dobara 255 se divide mat karein**.
- Sirf **3 epochs**. Accuracy ka koi target nahi, samajh aur sahi evaluation chahiye.
- Ye sirf ek saaf digit parhta hai, poori handwriting nahi.

In [ ]:
# Cell 12: Pehle kuch images aur labels dekhna
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(images["x_train"][i].squeeze(), cmap="gray")
    ax.set_title(meta["image_classes"][int(images["y_train"][i])])
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Pixel min/max:", images["x_train"].min(), images["x_train"].max())

In [ ]:
# Cell 13: CNN banana aur train karna
import tensorflow as tf
tf.keras.utils.set_random_seed(42)

image_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Rescaling(1./255),                       # pixels ko 0 se 1 me laata hai (sirf yahin, ek dafa)
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")])

image_model.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])

stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)

history = image_model.fit(
    images["x_train"], images["y_train"],
    validation_data=(images["x_validation"], images["y_validation"]),
    epochs=3, batch_size=32, callbacks=[stop])

image_model.save(PROJECT_DIR / "models/images.keras")
print("Saved: models/images.keras")

In [ ]:
# Cell 14: Training curves (accuracy aur loss)
h = history.history
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(h["accuracy"], marker="o", label="train")
axes[0].plot(h["val_accuracy"], marker="o", label="validation")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(h["loss"], marker="o", label="train")
axes[1].plot(h["val_loss"], marker="o", label="validation")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/04_cnn_curves.png", dpi=150)
plt.show()
# Agar train behtar ho aur validation bigre to overfitting ka imkaan hai, report me likhein.

In [ ]:
# Cell 15: CNN ka FINAL test evaluation (ek dafa)
probs = image_model.predict(images["x_test"], verbose=0)
cnn_pred = probs.argmax(axis=1)                  # sab se bada score wali class
y_test_img = images["y_test"]

cnn_metrics = {
    "accuracy": round(float(accuracy_score(y_test_img, cnn_pred)), 3),
    "macro_f1": round(float(f1_score(y_test_img, cnn_pred, average="macro")), 3),
}
print("CNN test:", cnn_metrics)

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_img, cnn_pred, display_labels=meta["image_classes"], ax=ax, colorbar=False)
ax.set_title("CNN: test confusion matrix")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/05_cnn_confusion.png", dpi=150)
plt.show()

# Teen galat predictions dikhana
wrong = np.where(cnn_pred != y_test_img)[0][:3]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 3))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, wrong):
        ax.imshow(images["x_test"][i].squeeze(), cmap="gray")
        ax.set_title(f"asli {y_test_img[i]}, model {cnn_pred[i]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

(PROJECT_DIR / "reports/cnn_metrics.json").write_text(json.dumps(cnn_metrics, indent=2))

In [ ]:
# Cell 16: App ke liye image ka preprocessing (sample image se test)
from PIL import Image

def prepare_image(file_or_path):
    img = Image.open(file_or_path).convert("L")     # grayscale (saved colour mode)
    img = img.resize((28, 28))                      # saved size
    arr = np.array(img, dtype="float32")            # 0 se 255 hi rehne dein
    return arr.reshape(1, 28, 28, 1)                # batch dimension

sample_path = PROJECT_DIR / "sample_images" / "3_3.png"
x = prepare_image(sample_path)
p = image_model.predict(x, verbose=0)[0]
print("Model ka andaza:", int(p.argmax()), "| score:", round(float(p.max()), 3), "(ye calibrated probability nahi hai)")
plt.imshow(x[0, :, :, 0], cmap="gray"); plt.axis("off"); plt.title("Processed image"); plt.show()

## Step 6: NLP, VADER sentiment aur topic rules
- 30 comments (`feedback.csv`), reference labels pehle se maujood hain. **Labels kabhi na badlein.**
- VADER: compound score >= 0.05 positive, <= -0.05 negative, warna neutral.
- Khali comment = `empty` (neutral nahi).
- Topic rules: lowercase karein, `topic_keywords.json` ki tarteeb me dekhein, pehla match jeete, warna `other`.

In [ ]:
# Cell 17: VADER chalana aur asli labels se muqabla
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.data.path.insert(0, str(PROJECT_DIR / "nltk_data"))   # offline lexicon, download ki zaroorat nahi
analyser = SentimentIntensityAnalyzer()

def sentiment(text):
    if pd.isna(text) or not str(text).strip():
        return "empty"
    score = analyser.polarity_scores(str(text))["compound"]
    return "positive" if score >= .05 else "negative" if score <= -.05 else "neutral"

feedback["predicted_sentiment"] = feedback["text"].map(sentiment)

labels = ["negative", "neutral", "positive"]
nlp_metrics = {
    "accuracy": round(accuracy_score(feedback["sentiment_label"], feedback["predicted_sentiment"]), 3),
    "macro_f1": round(f1_score(feedback["sentiment_label"], feedback["predicted_sentiment"],
                               labels=labels, average="macro"), 3),
    "confusion_matrix_order": labels,
    "confusion_matrix": confusion_matrix(feedback["sentiment_label"], feedback["predicted_sentiment"], labels=labels).tolist(),
}
print("VADER vs reference:", nlp_metrics)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    feedback["sentiment_label"], feedback["predicted_sentiment"], labels=labels, ax=ax, colorbar=False)
ax.set_title("VADER: confusion matrix")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/06_vader_confusion.png", dpi=150)
plt.show()

# Jahan model aur reference label me ikhtilaf hai (report me 3 tak explain karein)
disagree = feedback[feedback["predicted_sentiment"] != feedback["sentiment_label"]]
display(disagree[["comment_id", "text", "sentiment_label", "predicted_sentiment"]])
(PROJECT_DIR / "reports/nlp_metrics.json").write_text(json.dumps(nlp_metrics, indent=2))

In [ ]:
# Cell 18: Topic rules
topic_keywords = json.loads((DATA / "topic_keywords.json").read_text())

def find_topic(text):
    # lowercase, phir file ki tarteeb me pehla match; matched keyword bhi wapas deta hai
    t = str(text).lower()
    for topic, words in topic_keywords.items():
        for w in words:
            if w in t:
                return topic, w
    return "other", None

feedback[["topic", "matched_keyword"]] = feedback["text"].apply(lambda s: pd.Series(find_topic(s)))
display(feedback[["text", "predicted_sentiment", "topic", "matched_keyword"]].head(10))

In [ ]:
# Cell 19: Sentiment aur topic ke 2 charts
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

s_counts = feedback["predicted_sentiment"].value_counts().reindex(labels, fill_value=0)
axes[0].bar(s_counts.index, s_counts.values, color=["#E45756", "#9D9D9D", "#59A14F"])
axes[0].set_title("Sentiment counts (VADER)")

t_counts = feedback["topic"].value_counts()
axes[1].bar(t_counts.index, t_counts.values, color="#4C78A8")
axes[1].set_title("Topic counts (keyword rules)")

plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports/07_sentiment_topic_counts.png", dpi=150)
plt.show()

## Step 7: Aakhri check aur download
Ye cell dikhata hai ke saari files ban gayin ya nahi, aur models + reports ko ZIP bana kar download karwata hai.
Is ZIP ko Colab band karne se **pehle** download kar lein.

In [ ]:
# Cell 20: Saved files ki list aur ZIP download
import shutil
for folder in ["models", "reports"]:
    print(folder + "/")
    for p in sorted((PROJECT_DIR / folder).glob("*")):
        print("   ", p.name, f"({p.stat().st_size // 1024} KB)")

# ZIP bana kar download
shutil.make_archive("/content/project01_outputs", "zip", PROJECT_DIR, "models")
shutil.make_archive("/content/project01_reports", "zip", PROJECT_DIR, "reports")
from google.colab import files
files.download("/content/project01_outputs.zip")
files.download("/content/project01_reports.zip")

## Report ke liye yaad-dehani
- **Data:** 395 records (276 train / 59 validation / 60 test), 3 charts.
- **Tree vs Baseline:** accuracy, macro F1, class-1 precision aur recall, confusion matrix, test me har class ke examples, 5 galat predictions.
- **KMeans:** group size + original-unit averages ki table, scatter plot, group ke naam (averages dekh kar).
- **CNN:** training curves, test accuracy, macro F1, confusion matrix, 3 galat predictions.
- **NLP:** VADER vs reference labels (accuracy, macro F1, confusion matrix), 3 tak ikhtilaf explain karna.
- **Limitations:** chhota test set (60 rows), MNIST sirf saaf digits, VADER mixed opinions me ghalti karta hai, ye sab sirf classroom exercise hai.